# 01 - Exploracao de Dados

Objetivo deste notebook:
- explorar todos os datasets brutos do projeto por fonte
- entender estrutura, colunas, tipos, nulos e duplicatas
- identificar a granularidade de cada base
- registrar regras para o notebook 02 de limpeza e tratamento

Padrao de organizacao:
- cada fonte tem sua propria secao
- funcoes comuns ficam no inicio
- nenhuma limpeza definitiva deve acontecer aqui


## 1. Imports, caminhos e configuracao

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PASTA_DATASETS = Path('../datasets')
PASTA_IDH = PASTA_DATASETS / 'idh'
PASTA_CRIMES = PASTA_DATASETS /'crimes'
PASTA_POPULACAO = PASTA_DATASETS / 'populacao'
PASTA_EDUCACAO = PASTA_DATASETS / 'educacao'

ANO_REFERENCIA_IDH = 2010

CONFIG_CSV_PADRAO = {
    'sep': ';',
    'encoding': 'utf-8-sig',
    'decimal': ',',
}


## 2. Funcoes auxiliares para exploracao

In [2]:
def listar_csvs(pasta: Path) -> list[Path]:
    if not pasta.exists():
        return []
    return sorted(pasta.glob('*.csv'))


def detectar_separador(caminho: Path) -> str:
    primeira_linha = caminho.read_text(encoding='utf-8-sig', errors='ignore').splitlines()[0]
    return ';' if primeira_linha.count(';') > primeira_linha.count(',') else ','


def ler_csv_padrao(caminho: Path) -> pd.DataFrame:
    separador = detectar_separador(caminho)
    df = pd.read_csv(
        caminho,
        sep=separador,
        encoding='utf-8-sig',
        decimal=',',
        low_memory=False,
    )
    return df.dropna(axis=1, how='all')


def resumo_dataframe(df: pd.DataFrame, nome_dataset: str) -> None:
    print(f'Dataset: {nome_dataset}')
    print(f'Linhas: {df.shape[0]}')
    print(f'Colunas: {df.shape[1]}')
    print(f'Duplicatas exatas: {df.duplicated().sum()}')
    print('\nColunas:')
    for coluna in df.columns:
        print('-', coluna)


def diagnostico_qualidade(df: pd.DataFrame) -> None:
    print('Tipos de dados:')
    display(df.dtypes.to_frame('tipo'))
    print('Nulos por coluna:')
    display(df.isna().sum().sort_values(ascending=False).to_frame('qtd_nulos'))
    print('Amostra de estatisticas:')
    display(df.describe(include='all').transpose())


def explorar_arquivo_csv(caminho: Path, nome_dataset: str) -> pd.DataFrame:
    print(f'Arquivo: {caminho}')
    print(f'Separador detectado: {detectar_separador(caminho)!r}')
    df = ler_csv_padrao(caminho)
    display(df.head())
    resumo_dataframe(df, nome_dataset)
    diagnostico_qualidade(df)
    return df


## 3. Inventario de arquivos brutos

Esta celula mostra quais CSVs ja existem em cada fonte. Se uma pasta ainda nao existir, ela aparece vazia e sera preenchida depois.


In [3]:
inventario = {
    'idh': listar_csvs(PASTA_IDH),
    'crimes': listar_csvs(PASTA_CRIMES),
    'populacao': listar_csvs(PASTA_POPULACAO),
    'educacao': listar_csvs(PASTA_EDUCACAO),
}

for fonte, arquivos in inventario.items():
    print(f'{fonte}:')
    for arquivo in arquivos:
        print(f'  - {arquivo}')
    if not arquivos:
        print('  - nenhum CSV encontrado')


idh:
  - ../datasets/idh/data_idhm_2010.csv
crimes:
  - ../datasets/crimes/2016-2021.csv
populacao:
  - ../datasets/populacao/2016-2021.csv
educacao:
  - ../datasets/educacao/2017-2021idep.csv


# Fonte 1 - IDH Municipal 2010

## 4.1 Carregar IDH

In [4]:
arquivos_idh = listar_csvs(PASTA_IDH)
arquivo_idh = arquivos_idh[0]
arquivo_idh


PosixPath('../datasets/idh/data_idhm_2010.csv')

In [5]:
df_idh_raw = explorar_arquivo_csv(arquivo_idh, 'IDH municipal 2010')


Arquivo: ../datasets/idh/data_idhm_2010.csv
Separador detectado: ';'


,Territorialidades,IDHM 2010,IDHM Renda 2010,IDHM Longevidade 2010,IDHM Educação 2010
0,Brasil,"0,727",0.739,0.816,0.637
1,Aracaju (SE),"0,77",0.784,0.823,0.708
2,Belém (PA),"0,746",0.751,0.822,0.673
3,Belo Horizonte (MG),"0,81",0.841,0.856,0.737
4,Boa Vista (RR),"0,752",0.737,0.816,0.708


Dataset: IDH municipal 2010
Linhas: 31
Colunas: 5
Duplicatas exatas: 0

Colunas:
- Territorialidades
- IDHM 2010
- IDHM Renda 2010
- IDHM Longevidade 2010
- IDHM Educação 2010
Tipos de dados:


,tipo
Territorialidades,object
IDHM 2010,object
IDHM Renda 2010,float64
IDHM Longevidade 2010,float64
IDHM Educação 2010,float64


Nulos por coluna:


,qtd_nulos
IDHM Renda 2010,3
IDHM Longevidade 2010,3
IDHM Educação 2010,3
IDHM 2010,2
Territorialidades,0


Amostra de estatisticas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Territorialidades,31,31,Brasil,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
IDHM 2010,29,25,"0,727",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
IDHM Renda 2010,28.0,NaN,NaN,NaN,0.788786,0.050101,0.723,0.7405,0.778,0.84025,0.876
IDHM Longevidade 2010,28.0,NaN,NaN,NaN,0.833393,0.019623,0.798,0.82,0.8295,0.8475,0.873
IDHM Educação 2010,28.0,NaN,NaN,NaN,0.708393,0.044651,0.635,0.6775,0.7075,0.7375,0.805


## 4.2 Inspecao da territorialidade no IDH

In [6]:
coluna_territorialidade_idh = next(
    coluna for coluna in df_idh_raw.columns
    if coluna.strip().lower().startswith('territorialidade')
)

coluna_territorialidade_idh


'Territorialidades'

In [7]:
amostra_territorialidade_idh = pd.DataFrame({
    'territorialidade': df_idh_raw[coluna_territorialidade_idh],
    'nome_municipio_extraido': df_idh_raw[coluna_territorialidade_idh].astype(str).str.replace(r'\s*\([A-Z]{2}\)$', '', regex=True),
    'sigla_uf_extraida': df_idh_raw[coluna_territorialidade_idh].astype(str).str.extract(r'\(([A-Z]{2})\)$')[0],
})

display(amostra_territorialidade_idh.head(20))


,territorialidade,nome_municipio_extraido,sigla_uf_extraida
0,Brasil,Brasil,NaN
1,Aracaju (SE),Aracaju,SE
2,Belém (PA),Belém,PA
3,Belo Horizonte (MG),Belo Horizonte,MG
4,Boa Vista (RR),Boa Vista,RR
5,Brasília (DF),Brasília,DF
6,Campo Grande (MS),Campo Grande,MS
7,Cuiabá (MT),Cuiabá,MT
8,Curitiba (PR),Curitiba,PR
9,Florianópolis (SC),Florianópolis,SC


In [ ]:
df_chave_idh = df_idh_raw.copy()
df_chave_idh['ano'] = ANO_REFERENCIA_IDH

duplicatas_chave_idh = df_chave_idh.duplicated(subset=[coluna_territorialidade_idh, 'ano']).sum()
print(f'Duplicatas na chave temporaria territorialidade + ano: {duplicatas_chave_idh}')


## 4.3 Conclusoes preliminares do IDH

Pontos para levar ao notebook 02:
- ler como CSV com separador `;`, encoding `utf-8-sig` e decimal `,`
- remover colunas totalmente vazias
- renomear colunas para pt-br padronizado
- criar `ano = 2010`
- extrair `nome_municipio` e `sigla_uf`
- criar `nome_municipio_padronizado`
- preparar `codigo_municipio` por cruzamento com tabela auxiliar


# Fonte 2 - Crimes

## 5.1 Carregar crimes

In [9]:
arquivos_crimes = listar_csvs(PASTA_CRIMES)
arquivos_crimes


[PosixPath('../datasets/crimes/2016-2021.csv')]

In [10]:
if arquivos_crimes:
    arquivo_crimes = arquivos_crimes[0]
    df_crimes_raw = explorar_arquivo_csv(arquivo_crimes, 'crimes')
else:
    print('Nenhum CSV de crimes encontrado em datasets/crimes.')


Arquivo: ../datasets/crimes/2016-2021.csv
Separador detectado: ','


,quantidade_mortes_intervencao_policial_civil_fora_de_servico,quantidade_feminicidio,quantidade_mortes_intervencao_policial_militar_fora_de_servico,quantidade_furto_veiculos,quantidade_mortes_intervencao_policial_civil_em_servico,quantidade_estupro,quantidade_morte_policiais_civis_confronto_em_servico,quantidade_mortes_intervencao_policial_militar_em_servico,quantidade_mortes_policiais_confronto,quantidade_posse_uso_entorpecente,...,quantidade_lesao_corporal_morte,proporcao_mortes_intenvencao_policial_x_mortes_violentas_intencionais,quantidade_morte_policiais_militares_confronto_em_servico,quantidade_posse_ilegal_porte_ilegal_arma_de_fogo,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,grupo,quantidade_homicidio_doloso
0,0.0,NaN,2.0,305.0,0.0,155.0,0.0,57.0,5.0,NaN,...,2.0,11.0,2.0,NaN,AL,Alagoas,2704302,Maceió,Grupo 1,449
1,0.0,NaN,7.0,2820.0,1.0,458.0,0.0,23.0,8.0,NaN,...,15.0,3.0,1.0,NaN,CE,Ceará,2304400,Fortaleza,Grupo 1,965
2,0.0,NaN,2.0,338.0,1.0,25.0,0.0,6.0,0.0,NaN,...,3.0,14.0,0.0,NaN,ES,Espírito Santo,3205309,Vitória,Grupo 1,51
3,0.0,NaN,24.0,3733.0,0.0,161.0,0.0,76.0,3.0,NaN,...,14.0,16.0,0.0,NaN,GO,Goiás,5208707,Goiânia,Grupo 1,452
4,0.0,NaN,0.0,487.0,0.0,224.0,0.0,27.0,1.0,NaN,...,12.0,4.0,0.0,NaN,MA,Maranhão,2111300,São Luís,Grupo 1,498


Dataset: crimes
Linhas: 162
Colunas: 32
Duplicatas exatas: 0

Colunas:
- quantidade_mortes_intervencao_policial_civil_fora_de_servico
- quantidade_feminicidio
- quantidade_mortes_intervencao_policial_militar_fora_de_servico
- quantidade_furto_veiculos
- quantidade_mortes_intervencao_policial_civil_em_servico
- quantidade_estupro
- quantidade_morte_policiais_civis_confronto_em_servico
- quantidade_mortes_intervencao_policial_militar_em_servico
- quantidade_mortes_policiais_confronto
- quantidade_posse_uso_entorpecente
- quantidade_mortes_violentas_intencionais
- quantidade_morte_policiais_militares_fora_de_servico
- quantidade_morte_policiais_civis_fora_de_servico
- quantidade_latrocinio
- quantidade_porte_ilegal_arma_de_fogo
- quantidade_mortes_intervencao_policial
- ano
- quantidade_roubo_furto_veiculos
- quantidade_posse_ilegal_arma_de_fogo
- quantidade_lesao_corporal_dolosa_violencia_domestica
- quantidade_trafico_entorpecente
- quantidade_roubo_veiculos
- quantidade_lesao_corporal_

,tipo
quantidade_mortes_intervencao_policial_civil_fora_de_servico,float64
quantidade_feminicidio,float64
quantidade_mortes_intervencao_policial_militar_fora_de_servico,float64
quantidade_furto_veiculos,float64
quantidade_mortes_intervencao_policial_civil_em_servico,float64
quantidade_estupro,float64
quantidade_morte_policiais_civis_confronto_em_servico,float64
quantidade_mortes_intervencao_policial_militar_em_servico,float64
quantidade_mortes_policiais_confronto,float64
quantidade_posse_uso_entorpecente,float64


Nulos por coluna:


,qtd_nulos
quantidade_mortes_intervencao_policial_civil_fora_de_servico,124
quantidade_mortes_intervencao_policial_militar_fora_de_servico,123
quantidade_mortes_intervencao_policial_civil_em_servico,122
quantidade_mortes_intervencao_policial_militar_em_servico,120
quantidade_morte_policiais_civis_confronto_em_servico,112
quantidade_morte_policiais_civis_fora_de_servico,112
quantidade_morte_policiais_militares_confronto_em_servico,111
quantidade_morte_policiais_militares_fora_de_servico,110
quantidade_posse_ilegal_arma_de_fogo,89
quantidade_porte_ilegal_arma_de_fogo,78


Amostra de estatisticas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
quantidade_mortes_intervencao_policial_civil_fora_de_servico,38.0,NaN,NaN,NaN,1.473684,3.261046,0.0,0.0,0.0,1.0,16.0
quantidade_feminicidio,106.0,NaN,NaN,NaN,8.735849,8.454136,0.0,4.0,6.0,10.75,44.0
quantidade_mortes_intervencao_policial_militar_fora_de_servico,39.0,NaN,NaN,NaN,11.794872,31.26136,0.0,0.0,4.0,7.5,147.0
quantidade_furto_veiculos,154.0,NaN,NaN,NaN,3317.571429,7278.00552,183.0,639.0,1195.0,2792.5,44668.0
quantidade_mortes_intervencao_policial_civil_em_servico,40.0,NaN,NaN,NaN,1.75,3.499084,0.0,0.0,0.0,2.0,17.0
quantidade_estupro,160.0,NaN,NaN,NaN,492.38125,505.203911,25.0,195.0,343.5,611.5,2663.0
quantidade_morte_policiais_civis_confronto_em_servico,50.0,NaN,NaN,NaN,0.16,0.548095,0.0,0.0,0.0,0.0,3.0
quantidade_mortes_intervencao_policial_militar_em_servico,42.0,NaN,NaN,NaN,41.690476,60.830628,0.0,7.75,22.5,50.0,281.0
quantidade_mortes_policiais_confronto,157.0,NaN,NaN,NaN,4.101911,8.15647,0.0,0.0,1.0,4.0,55.0
quantidade_posse_uso_entorpecente,107.0,NaN,NaN,NaN,1211.065421,1900.215689,0.0,163.5,505.0,1440.0,11964.0


## 5.2 Pontos a decidir para crimes

Registrar aqui depois da exploracao:
- coluna de `codigo_municipio`
- coluna de `ano`
- coluna de quantidade de ocorrencias
- categorias de crime que serao usadas
- regra de agregacao para `municipio + ano`


# Fonte 3 - Populacao

In [11]:
arquivos_populacao = listar_csvs(PASTA_POPULACAO)
if arquivos_populacao:
    arquivo_populacao = arquivos_populacao[0]
    df_populacao_raw = explorar_arquivo_csv(arquivo_populacao, 'populacao')
else:
    print('Nenhum CSV de populacao encontrado em datasets/populacao.')


Arquivo: ../datasets/populacao/2016-2021.csv
Separador detectado: ','


,ano,id_municipio,id_municipio_nome,sexo,grupo_idade,populacao
0,2016,5107206,Rio Branco,feminino,0-4 anos,166
1,2016,2211001,Teresina,feminino,0-4 anos,26860
2,2016,1721000,Palmas,feminino,10-14 anos,13116
3,2016,4314902,Porto Alegre,feminino,10-14 anos,45184
4,2016,2502151,Boa Vista,feminino,10-14 anos,291


Dataset: populacao
Linhas: 6936
Colunas: 6
Duplicatas exatas: 0

Colunas:
- ano
- id_municipio
- id_municipio_nome
- sexo
- grupo_idade
- populacao
Tipos de dados:


,tipo
ano,int64
id_municipio,int64
id_municipio_nome,object
sexo,object
grupo_idade,object
populacao,int64


Nulos por coluna:


,qtd_nulos
ano,0
id_municipio,0
id_municipio_nome,0
sexo,0
grupo_idade,0
populacao,0


Amostra de estatisticas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,6936.0,NaN,NaN,NaN,2018.5,1.707948,2016.0,2017.0,2018.5,2020.0,2021.0
id_municipio,6936.0,NaN,NaN,NaN,2966262.823529,1225382.923338,1100205.0,2211001.0,2701155.5,4106902.0,5300108.0
id_municipio_nome,6936,27,Belém,612,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sexo,6936,2,feminino,3468,NaN,NaN,NaN,NaN,NaN,NaN,NaN
grupo_idade,6936,17,0-4 anos,408,NaN,NaN,NaN,NaN,NaN,NaN,NaN
populacao,6936.0,NaN,NaN,NaN,43274.798731,72917.699697,38.0,4270.25,21066.0,49269.75,548908.0


# Fonte 4 - Educacao

In [12]:
arquivos_educacao = listar_csvs(PASTA_EDUCACAO)
if arquivos_educacao:
    arquivo_educacao = arquivos_educacao[0]
    df_educacao_raw = explorar_arquivo_csv(arquivo_educacao, 'educacao')
else:
    print('Nenhum CSV de educacao encontrado em datasets/educacao.')


Arquivo: ../datasets/educacao/2017-2021idep.csv
Separador detectado: ';'


,ibge_id,dependencia_id,ciclo_id,ano,ideb,fluxo,aprendizado,nota_mt,nota_lp
0,11,0,EM,2017,4.0,0.8735,4.5271,272.03,268.33
1,11,2,EM,2017,3.8,0.8498,4.4200,268.05,264.91
2,11,4,EM,2017,5.5,0.9496,5.7899,319.00,308.62
3,12,0,EM,2017,3.8,0.8770,4.3461,263.28,264.45
4,12,2,EM,2017,3.6,0.8476,4.2562,260.01,261.51


Dataset: educacao
Linhas: 243
Colunas: 9
Duplicatas exatas: 0

Colunas:
- ibge_id
- dependencia_id
- ciclo_id
- ano
- ideb
- fluxo
- aprendizado
- nota_mt
- nota_lp
Tipos de dados:


,tipo
ibge_id,int64
dependencia_id,int64
ciclo_id,object
ano,int64
ideb,float64
fluxo,float64
aprendizado,float64
nota_mt,float64
nota_lp,float64


Nulos por coluna:


,qtd_nulos
ibge_id,0
dependencia_id,0
ciclo_id,0
ano,0
ideb,0
fluxo,0
aprendizado,0
nota_mt,0
nota_lp,0


Amostra de estatisticas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ibge_id,243.0,NaN,NaN,NaN,29.111111,12.807539,11.0,17.0,27.0,41.0,53.0
dependencia_id,243.0,NaN,NaN,NaN,2.0,1.636364,0.0,0.0,2.0,4.0,4.0
ciclo_id,243,1,EM,243,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ano,243.0,NaN,NaN,NaN,2019.0,1.636364,2017.0,2017.0,2019.0,2021.0,2021.0
ideb,243.0,NaN,NaN,NaN,4.457613,0.999614,2.7,3.6,4.1,5.5,6.4
fluxo,243.0,NaN,NaN,NaN,0.893899,0.070453,0.6926,0.8397,0.9032,0.95635,0.9912
aprendizado,243.0,NaN,NaN,NaN,4.940266,0.794323,3.5826,4.31245,4.7085,5.6817,6.6895
nota_mt,243.0,NaN,NaN,NaN,285.397325,30.762476,237.25,261.44,276.4,312.215,357.5
nota_lp,243.0,NaN,NaN,NaN,283.389918,24.387595,237.87,263.62,276.55,306.625,333.53


# Conclusao geral da exploracao

Ao final deste notebook, cada fonte deve ter uma lista clara de regras para o notebook 02:
- quais colunas serao mantidas
- quais colunas precisam ser renomeadas
- quais chaves serao usadas
- qual granularidade precisa ser atingida
- quais problemas de qualidade precisam ser tratados
